In [0]:
from pyspark.sql import functions as F, Window

CAT = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.silver")

DataFrame[]

In [0]:
df = spark.table(f"{CAT}.bronze.tb_movies_info")

# o bronze usa append, que aí o mesmo filme pode aparecer várias vezes.
df = df.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))

w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

In [0]:
# o status vem com ruído. Normaliza primeiro e só depois traduz
# o que não tiver no mapa vira "Não Informado"

# normaliza
df = df.withColumn(
    "status_norm",
    F.trim(F.regexp_replace(
        F.regexp_replace(
            F.regexp_replace(F.lower(F.col("status")), r"[-_]+", " "),
            r"[^a-z ]", ""),
        r"\s+", " "))
)

# traduz
mapa_status = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado",
}
status_pt = F.lit("Não Informado")
for k, v in mapa_status.items():
    status_pt = F.when(F.col("status_norm") == k, F.lit(v)).otherwise(status_pt)

df = df.withColumn("status_filme", status_pt)

In [0]:
# a data vem em 3 padrões
# try_to_date devolve NULL em vez de erro, então só vira NULL o que não tem conversão possível
formatos = ["yyyy-MM-dd",
            "dd/MM/yyyy",
            "MM-dd-yyyy"]

data_lancamento = F.coalesce(*[F.expr(f"try_to_date(trim(release_date), '{f}')") for f in formatos])

silver_info = (df
    .withColumn("data_lancamento", data_lancamento)
    .withColumn("ano_lancamento", F.year("data_lancamento"))
    .withColumn("duracao_minutos", F.expr("try_cast(try_cast(trim(runtime) as double) as int)"))
    .select(
        F.col("id").alias("id_filme"),
        F.col("title").alias("titulo"),
        F.col("original_title").alias("titulo_original"),
        "data_lancamento",
        "duracao_minutos",
        F.col("original_language").alias("idioma_original"),
        "status_filme",
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao"),
        "ano_lancamento",
    ))

(silver_info.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_info_filmes"))

In [ ]:
# conferências da tb_info_filmes
s = spark.table(f"{CAT}.silver.tb_info_filmes")
print("linhas:", s.count(), "| ids únicos:", s.select("id_filme").distinct().count())
s.groupBy("status_filme").count().show()

print("datas nulas:", s.filter("data_lancamento is null").count())

# valores originais que não converteram
chk = df.withColumn("data_lancamento", data_lancamento)
(chk.filter("data_lancamento is null and release_date is not null")
    .select("release_date").distinct().show(50, False))

# padrões de data que existem na base
(df.withColumn("padrao", F.regexp_replace(F.regexp_replace(F.trim("release_date"), r"[0-9]", "9"), r"[A-Za-z]", "a"))
   .groupBy("padrao").count().orderBy(F.desc("count")).show(20, False))

In [ ]:
# olhar como a cotação ficou na bronze antes de tratar
b = spark.table(f"{CAT}.bronze.tb_cotacao_dolar")
b.printSchema()
b.orderBy("dataHoraCotacao").show(15, False)
print("linhas:", b.count())

In [ ]:
# a API do BC só tem cotação em dia útil. a série precisa ser contínua
# fim de semana e feriado recebem a cotação do último dia útil
cot = spark.table(f"{CAT}.bronze.tb_cotacao_dolar")

cot = (cot
    .withColumn("data_cotacao", F.expr("try_to_date(substring(trim(dataHoraCotacao), 1, 10), 'yyyy-MM-dd')"))
    .withColumn("cotacao_dolar", F.expr("try_cast(cotacaoCompra as decimal(18,6))"))
    .filter("data_cotacao is not null and cotacao_dolar is not null and cotacao_dolar > 0"))

# a API devolve mais de um boletim por dia: fica só o último do dia
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())
cot = cot.withColumn("rn", F.row_number().over(w_dia)).filter("rn = 1").select("data_cotacao", "cotacao_dolar")

# calendário contínuo entre a primeira e a última cotação
calendario = (cot.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim"))
    .select(F.explode(F.sequence("ini", "fim", F.expr("interval 1 day"))).alias("data_cotacao")))

# forward fill: repete o último valor não nulo
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
silver_cot = (calendario.join(cot, "data_cotacao", "left")
    .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(w_ff)))

(silver_cot.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_cotacao_dolar"))

spark.table(f"{CAT}.silver.tb_cotacao_dolar").orderBy("data_cotacao").show(15)

In [ ]:
# budget/revenue vêm sujos
# limpa primeiro e só depois converte pra decimal. texto sem número vira NULL
fin = spark.table(f"{CAT}.bronze.tb_movies_financials")
fin = fin.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))

for origem, destino in [("budget", "orcamento_usd"), ("revenue", "receita_usd")]:
    fin = (fin
        .withColumn("_s", F.upper(F.trim(F.col(origem).cast("string"))))
        .withColumn("_num", F.regexp_replace("_s", r"[^0-9.\-]", ""))
        .withColumn(destino,
            F.expr("try_cast(_num as decimal(20,2))")
            * F.when(F.col("_s").rlike(r"^[0-9.]+M$"), 1000000).otherwise(1))   # 34.0M = 34 milhões
        # valor zerado ou negativo não faz sentido pra orçamento/receita: vira NULL
        .withColumn(destino, F.when(F.col(destino) > 0, F.col(destino).cast("decimal(18,2)")))
        .drop("_s", "_num"))

# 1 linha por filme. ingestion_datetime é igual nas repetidas, então desempata pela linha mais completa
completude = F.col("orcamento_usd").isNotNull().cast("int") + F.col("receita_usd").isNotNull().cast("int")
w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc(), completude.desc(),
                                     F.col("orcamento_usd").desc_nulls_last(), F.col("receita_usd").desc_nulls_last())
fin = fin.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

# regra: conversão pra BRL com a cotação mais recente da silver
cotacao = spark.table(f"{CAT}.silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()["cotacao_dolar"]
print("cotação usada:", cotacao)

silver_fin = (fin
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    # lucro: valor ausente conta como 0 pra não anular a conta; só fica NULL se orçamento E receita faltam
    .withColumn("lucro_usd",
        F.when(F.col("orcamento_usd").isNull() & F.col("receita_usd").isNull(), F.lit(None).cast("decimal(18,2)"))
         .otherwise((F.coalesce("receita_usd", F.lit(0)) - F.coalesce("orcamento_usd", F.lit(0))).cast("decimal(18,2)")))
    .withColumn("lucro_brl", F.round(F.col("lucro_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    # margem %: só calcula com receita > 0, o que evita divisão por zero
    .withColumn("margem_lucro_percentual",
        F.when(F.col("receita_usd") > 0, F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2).cast("double")))
    .select(F.col("id").alias("id_filme"), "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_percentual"))

(silver_fin.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_financeiro_filmes"))

In [ ]:
# conferências da tb_financeiro_filmes
s = spark.table(f"{CAT}.silver.tb_financeiro_filmes")
print("linhas:", s.count(), "| ids únicos:", s.select("id_filme").distinct().count())
s.printSchema()

# quantos nulos em cada coluna
s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns]).show()

# os casos que a gente viu na bronze (duplicado com $, N/A e 0)
s.filter(F.col("id_filme").isin("382589", "524369", "744276")).show(truncate=False)

s.orderBy(F.col("receita_usd").desc_nulls_last()).show(5, False)

In [ ]:
# as colunas vêm como texto sujo, só converte quando o texto é de fato um número. qualquer outra coisa vira NULL sem quebrar o pipeline
def limpa_num(coluna, virgula_decimal=False):
    s = f"trim(cast({coluna} as string))"
    if virgula_decimal:
        s = f"regexp_replace({s}, ',', '.')"
    return F.expr(f"CASE WHEN {s} rlike '^-?[0-9]+([.][0-9]+)?$' THEN try_cast({s} as decimal(20,6)) END")

def nota_0_a_10(coluna):
    v = limpa_num(coluna)
    # nota fora da escala é erro de escala: vira NULL
    return F.when((v >= 0) & (v <= 10), v.cast("double"))

def contagem_valida(coluna):
    v = limpa_num(coluna)
    # contagem de votos é inteira e não negativa, decimal quebrado é dado deslocado de outra coluna
    return F.when((v >= 0) & (v <= 2147483647) & (v == F.floor(v)), v.cast("int"))

met = spark.table(f"{CAT}.bronze.tb_movies_metrics")
met = met.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))

pop = limpa_num("popularity", virgula_decimal=True)
met = (met
    .withColumn("popularidade", F.when(pop >= 0, pop.cast("double")))   # negativo vira NULL
    .withColumn("nota_media_tmdb", nota_0_a_10("vote_average"))
    .withColumn("qtd_votos_tmdb", contagem_valida("vote_count"))
    .withColumn("nota_media_imdb", nota_0_a_10("averageRating"))
    .withColumn("qtd_votos_imdb", contagem_valida("numVotes")))

# 1 linha por filme. limpa primeiro e desempata pela linha mais completa
cols_m = ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]
completude = sum(F.col(c).isNotNull().cast("int") for c in cols_m)
w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc(), completude.desc(),
                                     *[F.col(c).desc_nulls_last() for c in cols_m])
met = met.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

silver_met = met.select(F.col("id").alias("id_filme"), *cols_m)

(silver_met.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_metricas_engajamento"))

In [ ]:
# conferências da tb_metricas_engajamento
s = spark.table(f"{CAT}.silver.tb_metricas_engajamento")
print("linhas:", s.count(), "| ids únicos:", s.select("id_filme").distinct().count())
s.printSchema()
s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns]).show()
s.select(F.min("popularidade"), F.max("popularidade"),
         F.min("nota_media_tmdb"), F.max("nota_media_tmdb"),
         F.min("nota_media_imdb"), F.max("nota_media_imdb"),
         F.min("qtd_votos_tmdb"), F.max("qtd_votos_tmdb"),
         F.min("qtd_votos_imdb"), F.max("qtd_votos_imdb")).show(truncate=False)
s.filter(F.col("id_filme").isin("299536", "475557")).show(truncate=False)

In [ ]:
# nota fora de 0 a 10 vira NULL, comentário vazio ou só com espaços vira "Sem comentário"
rev = spark.table(f"{CAT}.bronze.tb_movies_reviews")
nota = limpa_num("nota")

silver_rev = (rev
    .withColumn("id", F.trim(F.col("id").cast("string")))
    .filter(F.col("id").isNotNull() & (F.col("id") != ""))
    .withColumn("nota_usuario", F.when((nota >= 0) & (nota <= 10), nota.cast("double")))
    .withColumn("nome_usuario", F.trim("nome"))
    .withColumn("comentario_usuario",
        F.when(F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""), F.lit("Sem comentário"))
         .otherwise(F.trim(F.col("comentario"))))
    .select(F.col("id").alias("id_filme"), "nome_usuario", "nota_usuario", "comentario_usuario")
    # remove as avaliações duplicadas
    .dropDuplicates())

(silver_rev.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_avaliacoes_usuarios"))

In [ ]:
# conferências da tb_avaliacoes_usuarios
s = spark.table(f"{CAT}.silver.tb_avaliacoes_usuarios")
print("linhas:", s.count(), "| linhas distintas:", s.distinct().count())
s.printSchema()
s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns]).show()
s.select(F.min("nota_usuario"), F.max("nota_usuario")).show()
print("sem comentário:", s.filter(F.col("comentario_usuario") == "Sem comentário").count())
s.show(8, False)

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame

# textos que significam "sem dado" viram NULL antes de qualquer tratamento
AUSENTES = ["n/a", "na", "unknown", "não informado", "nao informado", "null", "none", "-", ""]
cols_cred = ["genres", "cast", "directors", "writers", "production_companies"]

cr = spark.table(f"{CAT}.bronze.tb_credits_and_tags")
cr = cr.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))
for c in cols_cred:
    cr = cr.withColumn(c, F.when(F.lower(F.trim(F.col(c))).isin(AUSENTES), F.lit(None).cast("string"))
                           .otherwise(F.trim(F.col(c))))

# 1 linha por filme, fica a ingestão mais recente e, no empate, a linha mais completa
completude = sum(F.col(c).isNotNull().cast("int") for c in cols_cred)
w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc(), completude.desc(),
                                     F.length(F.concat_ws("", *cols_cred)).desc())
cr = cr.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

# as colunas guardam vários valores na mesma célula, com separadores misturados, padroniza o separador, separa e limpa aspas/espaços
def explode_lista(df, coluna):
    itens = F.explode(F.split(F.regexp_replace(F.col(coluna), r"[;|]", ","), ","))
    return (df.select(F.col("id").alias("id_filme"), itens.alias("item"))
              .withColumn("item", F.trim(F.regexp_replace("item", r'[\\"]', "")))
              .withColumn("item", F.regexp_replace("item", r"\s+", " ")))

# gênero é um conjunto fechado (catálogo do TMDB). o que não pertence a esse domínio é descartado
GENEROS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family",
           "Fantasy", "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction",
           "TV Movie", "Thriller", "War", "Western"]
mapa_gen = {g.lower(): g for g in GENEROS}

genero = F.lit(None).cast("string")
for k, v in mapa_gen.items():
    genero = F.when(F.lower(F.col("item")) == k, F.lit(v)).otherwise(genero)

silver_gen = (explode_lista(cr, "genres")
    .withColumn("genero", genero)
    .filter(F.col("genero").isNotNull())
    .select("id_filme", "genero")
    .dropDuplicates())

(silver_gen.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_generos"))

In [ ]:
# conferências da tb_generos
s = spark.table(f"{CAT}.silver.tb_generos")
print("linhas:", s.count(), "| filmes com gênero:", s.select("id_filme").distinct().count())
s.groupBy("genero").count().orderBy(F.desc("count")).show(25, False)

# o que foi descartado
(explode_lista(cr, "genres").filter(~F.lower("item").isin(list(mapa_gen.keys())))
    .groupBy("item").count().orderBy(F.desc("count")).show(30, False))

In [ ]:
# cast, directors, writers e production_companies viram uma única tabela, com o tipo da entidade
tipos = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}
pe_bruto = reduce(DataFrame.unionByName,
    [explode_lista(cr, c).withColumn("tipo_entidade", F.lit(t)) for c, t in tipos.items()])

# descarta vazio, item sem nenhuma letra, "N/A"-like e texto longo demais pra ser nome
valido = (F.col("item").rlike(r"[A-Za-zÀ-ÿ]")
          & ~F.lower(F.col("item")).isin(AUSENTES)
          & (F.length("item") <= 80))

# padroniza a capitalização quando o texto está todo minúsculo ou todo maiúsculo, nomes com caixa mista ficam como estão. siglas curtas também
nome = (F.when((F.col("item") == F.lower("item")) |
               ((F.col("item") == F.upper("item")) & (F.length("item") > 4)), F.initcap("item"))
         .otherwise(F.col("item")))

silver_pe = (pe_bruto.filter(valido)
    .select("id_filme", nome.alias("nome_entidade"), "tipo_entidade")
    .dropDuplicates())   # remove repetição do mesmo nome/tipo no mesmo filme

(silver_pe.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_pessoas_empresas"))

In [ ]:
# conferências da tb_pessoas_empresas
s = spark.table(f"{CAT}.silver.tb_pessoas_empresas")
print("linhas:", s.count())
s.groupBy("tipo_entidade").agg(F.count("*").alias("linhas"),
                               F.countDistinct("nome_entidade").alias("nomes_distintos")).show()

# mais frequentes de cada tipo. lixo de column shift costuma aparecer aqui com contagem alta
for t in ["Ator", "Diretor", "Roteirista", "Produtora"]:
    print(t)
    s.filter(F.col("tipo_entidade") == t).groupBy("nome_entidade").count().orderBy(F.desc("count")).show(12, False)

# nomes de pessoa com dígito (suspeito)
(s.filter((F.col("tipo_entidade") != "Produtora") & F.col("nome_entidade").rlike(r"[0-9]"))
   .groupBy("nome_entidade").count().orderBy(F.desc("count")).show(15, False))

# oq foi descartado
print("descartados")
(pe_bruto.filter(~valido).groupBy("tipo_entidade", "item").count().orderBy(F.desc("count")).show(20, False))